# 🚀 SOTA E-Commerce Visual Search Engine (Colab Edition)

Notebook ini akan mengunduh repositori arsitektur mesin pencari canggih (FastAPI + Qdrant + CLIP + React) yang telah kita bangun, mengonfigurasi environment di Google Colab, menjalankan server backend & frontend di latar belakang, dan memberikan Anda URL publik melalui **Localtunnel** agar Anda bisa langsung menguji purwarupa UI aplikasi ini.

---

## Langkah 1: Persiapan Environment & Download Source Code

In [1]:
!pip install --upgrade numpy pandas


!rm -rf Smart-Catalog-Amazon-Berkeley-Objects
!git clone https://github.com/Brian071/Smart-Catalog-Amazon-Berkeley-Objects.git


# Membuat direktori dataset utama
!mkdir -p dataset

# Mengunduh dataset Listings dan Images langsung dari server AWS S3
!wget -q https://amazon-berkeley-objects.s3.amazonaws.com/archives/abo-listings.tar -O dataset/abo-listings.tar
!wget -q https://amazon-berkeley-objects.s3.amazonaws.com/archives/abo-images-small.tar -O dataset/abo-images-small.tar

# Mengekstrak isi dataset ke dalam folder penampung
!tar -xf dataset/abo-listings.tar -C dataset
!tar -xf dataset/abo-images-small.tar -C dataset

# Uninstall paket yang berpotensi konflik dari memori Colab
!pip uninstall -y transformers torch torchvision numpy scipy pandas seaborn

# Install seluruh dependensi secara aman dengan membatasi versi NumPy di bawah 2.0
!pip install "numpy<2.0.0" scipy pandas seaborn fastapi uvicorn torch torchvision transformers[torch] qdrant-client>=1.10.0 Pillow "opencv-python<4.11.0" optuna segment-anything python-multipart urllib3>=2 matplotlib

# Install Localtunnel
!npm install -g localtunnel

Cloning into 'Smart-Catalog-Amazon-Berkeley-Objects'...
remote: Enumerating objects: 92, done.
remote: Counting objects: 100% (92/92), done.
remote: Compressing objects: 100% (81/81), done.
remote: Total 92 (delta 31), reused 52 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (92/92), 264.42 KiB | 4.13 MiB/s, done.
Resolving deltas: 100% (31/31), done.
Found existing installation: transformers 5.12.1
Uninstalling transformers-5.12.1:
  Successfully uninstalled transformers-5.12.1
Found existing installation: torch 2.11.0+cpu
Uninstalling torch-2.11.0+cpu:
  Successfully uninstalled torch-2.11.0+cpu
Found existing installation: torchvision 0.26.0+cpu
Uninstalling torchvision-0.26.0+cpu:
  Successfully uninstalled torchvision-0.26.0+cpu
Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
Found existing installation: scipy 1.16.3
Uninstalling scipy-1.16.3:
  Successfully uninstalled scipy-1.16.3
Found existing installation: p

## Langkah 2: Memulai Server Backend (FastAPI)

In [2]:
import os
import subprocess
import time
import requests

# Kill any existing processes on port 8000 that might be lingering from previous runs
!fuser -k 8000/tcp || true

# Jalankan Backend di background
# Temporarily remove stdout/stderr pipes to see output directly for debugging
backend_process = subprocess.Popen(
    ["uvicorn", "api:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="/content/Smart-Catalog-Amazon-Berkeley-Objects/backend"
    # stdout=subprocess.PIPE,  # Removed for debugging
    # stderr=subprocess.PIPE   # Removed for debugging
)
print("Memulai Backend FastAPI di port 8000...")

# Add a robust health check
backend_ready = False
max_retries = 60 # Try for up to 60 * 1 second = 60 seconds (increased from 30)
for i in range(max_retries):
    try:
        response = requests.get("http://localhost:8000/", timeout=1)
        if response.status_code == 200:
            print("Backend FastAPI siap!")
            backend_ready = True
            break
    except requests.exceptions.ConnectionError:
        pass # Server not ready yet
    print(f"Waiting for backend... ({i+1}/{max_retries})")
    time.sleep(1)

if not backend_ready:
    print("\n=======================================================")
    print("Gagal memulai Backend FastAPI. Harap periksa log server untuk kesalahan.")
    # If pipes are not used, no need to communicate. The errors should have been printed directly by uvicorn.
    print("Check the output above for any Uvicorn/FastAPI errors.")
    print("=======================================================\n")
    backend_process.kill() # Ensure it's stopped if it failed
    raise RuntimeError("Backend FastAPI did not start within the expected time.")

Memulai Backend FastAPI di port 8000...
Waiting for backend... (1/60)
Waiting for backend... (2/60)
Waiting for backend... (3/60)
Waiting for backend... (4/60)
Waiting for backend... (5/60)
Waiting for backend... (6/60)
Waiting for backend... (7/60)
Waiting for backend... (8/60)
Waiting for backend... (9/60)
Waiting for backend... (10/60)
Waiting for backend... (11/60)
Waiting for backend... (12/60)
Waiting for backend... (13/60)
Waiting for backend... (14/60)
Waiting for backend... (15/60)
Waiting for backend... (16/60)
Waiting for backend... (17/60)
Waiting for backend... (18/60)
Waiting for backend... (19/60)
Waiting for backend... (20/60)
Waiting for backend... (21/60)
Waiting for backend... (22/60)
Waiting for backend... (23/60)
Waiting for backend... (24/60)
Waiting for backend... (25/60)
Waiting for backend... (26/60)
Waiting for backend... (27/60)
Waiting for backend... (28/60)
Waiting for backend... (29/60)
Waiting for backend... (30/60)
Waiting for backend... (31/60)
Waiting 

In [3]:
import os
import re

print("1. Mematikan server agar tidak terjadi bentrok data (File Lock)...")
os.system("fuser -k 8000/tcp || true")
os.system("pkill -f cloudflared || true")

print("2. Memaksa Qdrant menyimpan data secara FISIK (Bukan RAM)...")
core_path = "/content/Smart-Catalog-Amazon-Berkeley-Objects/backend/core.py"
with open(core_path, "r") as f:
    core_text = f.read()

# Mengubah :memory: menjadi penyimpanan folder lokal
core_text = re.sub(
    r'QdrantClient\([^\)]*\)',
    'QdrantClient(path="/content/Smart-Catalog-Amazon-Berkeley-Objects/backend/qdrant_storage")',
    core_text
)
with open(core_path, "w") as f:
    f.write(core_text)

print("3. Menghapus saringan kemiripan di api.py...")
api_path = "/content/Smart-Catalog-Amazon-Berkeley-Objects/backend/api.py"
with open(api_path, "r") as f:
    api_text = f.read()
api_text = api_text.replace('threshold=TRIAL_228_CONFIG["similarity_threshold"]', 'threshold=None')
with open(api_path, "w") as f:
    f.write(api_text)

print("\n✅ TAHAP 1 SELESAI. SERVER TELAH DIMATIKAN DAN KODE DIPERBARUI.")
print("👉 SEKARANG: Gulir ke atas dan jalankan ulang SEL LANGKAH 3 (Indeksasi 761 gambar).")

1. Mematikan server agar tidak terjadi bentrok data (File Lock)...
2. Memaksa Qdrant menyimpan data secara FISIK (Bukan RAM)...
3. Menghapus saringan kemiripan di api.py...

✅ TAHAP 1 SELESAI. SERVER TELAH DIMATIKAN DAN KODE DIPERBARUI.
👉 SEKARANG: Gulir ke atas dan jalankan ulang SEL LANGKAH 3 (Indeksasi 761 gambar).


## Langkah 3: Mengindeks Dataset ABO untuk Testing
Mari kita muat beberapa sampel dataset Amazon Berkeley Objects (ABO) ke dalam Qdrant agar bisa langsung dicari.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 31.1 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.5.0 which is incompatible.


In [1]:
import requests
from PIL import Image
import io
import json
import os
import gzip
import shutil

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# --- Step 1 & 2: Load and prepare metadata using the user's provided logic ---
print("Memuat metadata produk...")
data = []
listing_path = 'dataset/listings/listings/metadata/listings_0.json.gz'

if not os.path.exists(listing_path):
    listing_path = 'dataset/listings/metadata/listings_0.json.gz'

if not os.path.exists(listing_path):
    print(f"Warning: Listing metadata file not found at {listing_path}. Please ensure 'dataset' directory is correctly set up.")
    listing_path = None

try:
    if listing_path:
        with gzip.open(listing_path, 'rt') as f:
            for i, line in enumerate(f):
                if i >= 30000:
                    break
                try:
                    item = json.loads(line.strip())
                    data.append(item)
                except json.JSONDecodeError as e:
                    continue
except FileNotFoundError:
    print(f"Error: File tidak ditemukan di {listing_path}. Pastikan proses download dan ekstrak di cell sebelumnya selesai.")
except Exception as e:
    print(f"An error occurred while loading listings metadata: {e}")

df_listings = pd.DataFrame(data)

def extract_en_text(item_list):
    if isinstance(item_list, list):
        for entry in item_list:
            if isinstance(entry, dict) and entry.get('language_tag', '').startswith('en_'):
                return entry.get('value', '')
        if len(item_list) > 0 and isinstance(item_list[0], dict):
            return item_list[0].get('value', '')
    return None

if not df_listings.empty:
    df_listings['item_name_en'] = df_listings['item_name'].apply(extract_en_text)
    if 'product_type' in df_listings.columns:
        df_listings['category'] = df_listings['product_type'].apply(lambda x: x[0]['value'] if isinstance(x, list) and len(x) > 0 else None)
    else:
        df_listings['category'] = None

print("Memuat metadata gambar...")
images_csv_path = 'dataset/images/metadata/images.csv.gz'
if not os.path.exists(images_csv_path):
    print(f"Warning: Image metadata file not found at {images_csv_path}. Please ensure 'dataset' directory is correctly set up.")
    df_images = pd.DataFrame()
else:
    try:
        df_images = pd.read_csv(images_csv_path)
    except Exception as e:
        print(f"Error loading images.csv.gz: {e}. Creating empty DataFrame.")
        df_images = pd.DataFrame()


df_merged = pd.DataFrame()
if not df_listings.empty and not df_images.empty:
    print("Menggabungkan data produk dan gambar...")
    df_merged = pd.merge(df_listings, df_images, left_on='main_image_id', right_on='image_id', how='inner')

    if 'path' in df_merged.columns:
        df_merged['image_path'] = 'dataset/images/' + df_merged['path']
    else:
        df_merged['image_path'] = None

    df_final = df_merged[['item_id', 'image_path', 'item_name_en', 'category']].rename(columns={
        'item_id': 'id',
        'item_name_en': 'text'
    })
else:
    print("Tidak dapat menggabungkan karena satu atau lebih DataFrame kosong.")
    df_final = pd.DataFrame()

if not df_final.empty:
    print("Preview data yang akan diindeks:")
    print(df_final.head())
else:
    print("df_final kosong, tidak ada data untuk diindeks.")


## --- Step 3: Merge, Filter for shoe items, and Index the data ---
df_merged = pd.DataFrame()
if not df_listings.empty and not df_images.empty:
    print("Menggabungkan data produk dan gambar...")
    df_merged = pd.merge(df_listings, df_images, left_on='main_image_id', right_on='image_id', how='inner')

    # PERBAIKAN: Menambahkan folder 'small/' agar sesuai dengan struktur asli dataset
    if 'path' in df_merged.columns:
        df_merged['image_path'] = 'dataset/images/small/' + df_merged['path']
    else:
        df_merged['image_path'] = None

    df_final = df_merged[['item_id', 'image_path', 'item_name_en', 'category']].rename(columns={
        'item_id': 'id',
        'item_name_en': 'text'
    })
else:
    print("Tidak dapat menggabungkan karena satu atau lebih DataFrame kosong.")
    df_final = pd.DataFrame()

print("\nMulai memfilter dan mengindeks gambar sepatu ke Vector DB...")
shoe_items_df = pd.DataFrame()
if not df_final.empty and 'category' in df_final.columns:
    shoe_items_df = df_final[
        (df_final['category'].str.contains('shoes', case=False, na=False)) |
        (df_final['category'].str.contains('footwear', case=False, na=False))
    ].copy()

if shoe_items_df.empty:
    print("Tidak ditemukan item sepatu setelah filtering.")
    total_items_to_process = 0
else:
    print(f"Ditemukan {len(shoe_items_df)} item sepatu untuk diindeks.")
    total_items_to_process = len(shoe_items_df)

indexed_count = 0
failed_count = 0

if total_items_to_process > 0:
    for i, row in shoe_items_df.iterrows():
        img_id = str(row['id'])
        image_local_path = row['image_path']

        if (i + 1) % 100 == 0 or i == 0 or (i + 1) == total_items_to_process:
            print(f"Processing item {i+1}/{total_items_to_process}: ID={img_id}")

        try:
            if not os.path.exists(image_local_path):
                print(f"Error: Gambar tidak ditemukan di {image_local_path}")
                failed_count += 1
                continue

            with open(image_local_path, 'rb') as f:
                img_bytes_content = f.read()
            img_bytes = io.BytesIO(img_bytes_content)

            files = {'file': (f'{img_id}.jpg', img_bytes, 'image/jpeg')}

            # PERBAIKAN: Hanya mengirim 'id' agar sesuai dengan kebutuhan Form di FastAPI
            payload_data = {'id': img_id}

            res = requests.post("http://localhost:8000/index_image", files=files, data=payload_data)
            res.raise_for_status()
            indexed_count += 1

        except requests.exceptions.RequestException as req_err:
            print(f"Gagal memposting {img_id}: {req_err}")
            if hasattr(req_err, 'response') and req_err.response is not None:
                print(f"Detail: {req_err.response.text}")
            failed_count += 1
        except Exception as e:
            print(f"Gagal memproses {img_id}: {e}")
            failed_count += 1
else:
    print("Tidak ada data untuk diindeks.")

print(f"\n--- Indeksasi Selesai ---")
print(f"Berhasil diindeks: {indexed_count}")
print(f"Gagal diindeks: {failed_count}")

Memuat metadata produk...
Memuat metadata gambar...
Menggabungkan data produk dan gambar...
Preview data yang akan diindeks:
           id                      image_path  \
0  B06X9STHNG  dataset/images/8c/8ccb5859.jpg   
1  B07P8ML82R  dataset/images/9f/9f76d27b.jpg   
2  B07H9GMYXS  dataset/images/66/665cc994.jpg   
3  B07CTPR73M  dataset/images/b4/b4f9d0cc.jpg   
4  B01MTEI8M6  dataset/images/2b/2b1c2516.jpg   

                                                text               category  
0  Amazon-merk - vinden. Dames Leder Gesloten Tee...                  SHOES  
1  22" Bottom Mount Drawer Slides, White Powder C...               HARDWARE  
2  AmazonBasics PETG 3D Printer Filament, 1.75mm,...  MECHANICAL_COMPONENTS  
3       Stone & Beam Stone Brown Swatch, 25020039-01                   SOFA  
4  The Fix Amazon Brand Women's French Floral Emb...                  SHOES  
Menggabungkan data produk dan gambar...

Mulai memfilter dan mengindeks gambar sepatu ke Vector DB...
Ditemukan 

## Langkah 4: Menjalankan Frontend React & Ekspos ke Internet

Karena arsitektur React tidak dirancang untuk ditenagai langsung melalui Cell IPython, kita akan menjalankan proses Node JS di latar belakang dan mempublikasikan URL-nya.

**Penting:** Klik link Localtunnel yang muncul pada output di bawah untuk mengakses aplikasi!

In [ ]:
# Install module Frontend
!cd /content/Smart-Catalog-Amazon-Berkeley-Objects/frontend && npm install

# Build Frontend (Pastikan file App.tsx sudah diisi dengan URL Cloudflare Backend port 8000)
!cd /content/Smart-Catalog-Amazon-Berkeley-Objects/frontend && npm run build

# Install server statis sederhana Python dan jalankan di background
import subprocess
frontend_process = subprocess.Popen(
    ["python3", "-m", "http.server", "3000"],
    cwd="/content/Smart-Catalog-Amazon-Berkeley-Objects/frontend/build"
)
print("Frontend berjalan di port 3000...")

# Membuka jalur Cloudflare kedua khusus untuk Frontend
print("\n=======================================================")
print("MENDAPATKAN PUBLIC URL UNTUK FRONTEND DARI CLOUDFLARE...")
print("Klik tautan berakhiran .trycloudflare.com di bawah ini.")
print("=======================================================\n")
!cloudflared tunnel --url http://localhost:3000

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸
up to date, audited 1322 packages in 6s
⠼
⠼272 packages are looking for funding
⠼  run `npm fund` for details
⠼
28 vulnerabilities (9 low, 6 moderate, 13 high)

To address issues that do not require attention, run:
  npm audit fix

To address all issues (including breaking changes), run:
  npm audit fix --force

Run `npm audit` for details.
⠴
> frontend@0.1.0 build
> react-scripts build

Creating an optimized production build...
Compiled successfully.

File sizes after gzip:

  62.95 kB  build/static/js/main.e4ddfc6c.js
  1.76 kB   build/static/js/453.20359781.chunk.js
  513 B     build/static/css/main.f855e6bc.css

The project was built assuming it is hosted at /.
You can control this with the homepage field in your package.json.

The build folder is ready to be deployed.
You may serve it with a static server:

  npm install -g serve
  serve -s build

Find out more about deployment here:

  https://cra.link/deployment

⠙Frontend be

In [4]:
import os
import subprocess
import time
import re

print("1. Menyalakan Server (Membaca data dari qdrant_storage)...")
uvicorn_log = open("uvicorn.log", "w")
subprocess.Popen(
    ["uvicorn", "api:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="/content/Smart-Catalog-Amazon-Berkeley-Objects/backend",
    stdout=uvicorn_log,
    stderr=uvicorn_log
)
time.sleep(5)

print("2. Membuka jalur Cloudflare...")
cf_log_file = "cloudflare.log"
if os.path.exists(cf_log_file):
    os.remove(cf_log_file)

cf_log = open(cf_log_file, "w")
subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=cf_log,
    stderr=cf_log
)

print("3. Menunggu URL dari Cloudflare...")
url = ""
for _ in range(30):
    time.sleep(1)
    with open(cf_log_file, "r") as f:
        content = f.read()
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', content)
        if match:
            url = match.group(0)
            break

if url:
    print("\n" + "="*70)
    print("✅ SERVER AKTIF! DATA 761 SEPATU TELAH TERBACA!")
    print(f"👉 KLIK URL INI UNTUK MEMBUKA WEB ANDA: {url}")
    print("⚠️ JANGAN TEKAN STOP. Biarkan sel ini terus berputar di Colab.")
    print("="*70 + "\n")
    try:
        while True:
            time.sleep(60)
    except KeyboardInterrupt:
        print("\nServer dihentikan.")
else:
    print("❌ Gagal mendapatkan URL. Periksa log.")

1. Menyalakan Server (Membaca data dari qdrant_storage)...
2. Membuka jalur Cloudflare...


FileNotFoundError: [Errno 2] No such file or directory: 'cloudflared'

In [14]:
%%writefile /content/Smart-Catalog-Amazon-Berkeley-Objects/backend/core.py
import torch
from transformers import CLIPProcessor, CLIPModel
import numpy as np
from PIL import Image
import cv2
import uuid
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams, PointStruct
import os

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

class Embedder:
    def __init__(self, model_id="openai/clip-vit-base-patch32"):
        self.model = CLIPModel.from_pretrained(model_id).to(device)
        self.processor = CLIPProcessor.from_pretrained(model_id)

    def get_text_embedding(self, text):
        inputs = self.processor(text=[text], return_tensors="pt", padding=True, truncation=True, max_length=77).to(device)
        with torch.no_grad():
            outputs = self.model.get_text_features(**inputs)
            text_features = outputs[0] if not isinstance(outputs, torch.Tensor) else outputs
        text_features = text_features / text_features.norm(p=2, dim=-1, keepdim=True)
        return [float(x) for x in text_features[0].cpu().numpy().flatten().tolist()[:512]]

    def get_image_embedding(self, image: Image.Image):
        inputs = self.processor(images=image, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = self.model.get_image_features(**inputs)
            image_features = outputs[0] if not isinstance(outputs, torch.Tensor) else outputs
        image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)
        return [float(x) for x in image_features[0].cpu().numpy().flatten().tolist()[:512]]

class VectorStore:
    def __init__(self, host="localhost", port=6333, collection_name="shoes"):
        storage_path = "/content/Smart-Catalog-Amazon-Berkeley-Objects/backend/qdrant_storage"
        os.makedirs(storage_path, exist_ok=True)
        self.client = QdrantClient(path=storage_path)
        self.collection_name = collection_name
        self.setup_collection()

    def setup_collection(self):
        try:
            self.client.get_collection(collection_name=self.collection_name)
        except Exception:
            self.client.create_collection(
                collection_name=self.collection_name,
                vectors_config=VectorParams(size=512, distance=Distance.COSINE),
            )

    def insert_image(self, id: str, vector: list, payload: dict = None):
        uuid_id = str(uuid.uuid5(uuid.NAMESPACE_DNS, id))
        self.client.upsert(
            collection_name=self.collection_name,
            points=[PointStruct(id=uuid_id, vector=vector, payload={"original_id": id, **(payload or {})})]
        )

    def search(self, query_vector: list, limit=5, threshold=None):
        try:
            return self.client.search(
                collection_name=self.collection_name,
                query_vector=query_vector,
                limit=limit,
                score_threshold=threshold
            )
        except Exception as e:
            return []

class ImageSegmenter:
    def __init__(self): pass
    def segment_shoe(self, image: Image.Image) -> Image.Image:
        cv_img = np.array(image.convert("RGB"))
        h, w = cv_img.shape[:2]
        mask = np.zeros(cv_img.shape[:2], np.uint8)
        bgdModel = np.zeros((1,65),np.float64)
        fgdModel = np.zeros((1,65),np.float64)
        rect = (int(w*0.1), int(h*0.1), int(w*0.8), int(h*0.8))
        cv2.grabCut(cv_img, mask, rect, bgdModel, fgdModel, 5, cv2.GC_INIT_WITH_RECT)
        mask2 = np.where((mask==2)|(mask==0), 0, 1).astype('uint8')
        img_segmented = cv_img * mask2[:, :, np.newaxis]
        white_bg = np.ones_like(cv_img) * 255
        white_bg_masked = white_bg * (1 - mask2[:, :, np.newaxis])
        return Image.fromarray((img_segmented + white_bg_masked).astype(np.uint8))

Overwriting /content/Smart-Catalog-Amazon-Berkeley-Objects/backend/core.py


In [15]:
%%writefile /content/Smart-Catalog-Amazon-Berkeley-Objects/backend/core.py
import torch
from transformers import CLIPProcessor, CLIPModel
import numpy as np
from PIL import Image
import cv2
import uuid
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams, PointStruct
import os

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

class Embedder:
    def __init__(self, model_id="openai/clip-vit-base-patch32"):
        self.model = CLIPModel.from_pretrained(model_id).to(device)
        self.processor = CLIPProcessor.from_pretrained(model_id)

    def get_text_embedding(self, text):
        inputs = self.processor(text=[text], return_tensors="pt", padding=True, truncation=True, max_length=77).to(device)
        with torch.no_grad():
            outputs = self.model.get_text_features(**inputs)
            text_features = outputs[0] if not isinstance(outputs, torch.Tensor) else outputs
        text_features = text_features / text_features.norm(p=2, dim=-1, keepdim=True)
        return [float(x) for x in text_features[0].cpu().numpy().flatten().tolist()[:512]]

    def get_image_embedding(self, image: Image.Image):
        inputs = self.processor(images=image, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = self.model.get_image_features(**inputs)
            image_features = outputs[0] if not isinstance(outputs, torch.Tensor) else outputs
        image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)
        return [float(x) for x in image_features[0].cpu().numpy().flatten().tolist()[:512]]

class VectorStore:
    def __init__(self, host="localhost", port=6333, collection_name="shoes"):
        storage_path = "/content/Smart-Catalog-Amazon-Berkeley-Objects/backend/qdrant_storage"
        os.makedirs(storage_path, exist_ok=True)
        self.client = QdrantClient(path=storage_path)
        self.collection_name = collection_name
        self.setup_collection()

    def setup_collection(self):
        try:
            self.client.get_collection(collection_name=self.collection_name)
        except Exception:
            self.client.create_collection(
                collection_name=self.collection_name,
                vectors_config=VectorParams(size=512, distance=Distance.COSINE),
            )

    def insert_image(self, id: str, vector: list, payload: dict = None):
        uuid_id = str(uuid.uuid5(uuid.NAMESPACE_DNS, id))
        self.client.upsert(
            collection_name=self.collection_name,
            points=[PointStruct(id=uuid_id, vector=vector, payload={"original_id": id, **(payload or {})})]
        )

    def search(self, query_vector: list, limit=5, threshold=None):
        try:
            return self.client.search(
                collection_name=self.collection_name,
                query_vector=query_vector,
                limit=limit,
                score_threshold=threshold
            )
        except Exception as e:
            return []

class ImageSegmenter:
    def __init__(self): pass
    def segment_shoe(self, image: Image.Image) -> Image.Image:
        cv_img = np.array(image.convert("RGB"))
        h, w = cv_img.shape[:2]
        mask = np.zeros(cv_img.shape[:2], np.uint8)
        bgdModel = np.zeros((1,65),np.float64)
        fgdModel = np.zeros((1,65),np.float64)
        rect = (int(w*0.1), int(h*0.1), int(w*0.8), int(h*0.8))
        cv2.grabCut(cv_img, mask, rect, bgdModel, fgdModel, 5, cv2.GC_INIT_WITH_RECT)
        mask2 = np.where((mask==2)|(mask==0), 0, 1).astype('uint8')
        img_segmented = cv_img * mask2[:, :, np.newaxis]
        white_bg = np.ones_like(cv_img) * 255
        white_bg_masked = white_bg * (1 - mask2[:, :, np.newaxis])
        return Image.fromarray((img_segmented + white_bg_masked).astype(np.uint8))

Overwriting /content/Smart-Catalog-Amazon-Berkeley-Objects/backend/core.py


In [3]:
%%writefile /content/Smart-Catalog-Amazon-Berkeley-Objects/frontend/src/App.tsx
import React, { useState } from 'react';
import './App.css';

const API_BASE_URL = '';

function App() {
  const [query, setQuery] = useState('');
  const [expandedQuery, setExpandedQuery] = useState('');
  const [suggestion, setSuggestion] = useState('');
  const [results, setResults] = useState<any[]>([]);
  const [loading, setLoading] = useState(false);
  const [baseImageId, setBaseImageId] = useState('');
  const [addText, setAddText] = useState('');
  const [subtractText, setSubtractText] = useState('');

  const executeSemanticSearch = async (searchQuery: string) => {
    setLoading(true); setSuggestion('');
    try {
      const response = await fetch(`${API_BASE_URL}/search/semantic?query=${encodeURIComponent(searchQuery)}`);
      const data = await response.json();
      setExpandedQuery(data.expanded_query);
      setResults(data.results);
      if (data.has_typo && data.corrected_query) setSuggestion(data.corrected_query);
    } catch (error) {
      alert("Gagal mengambil data pencarian semantik.");
    }
    setLoading(false);
  };

  const handleCompositionalSearch = async () => {
    setLoading(true);
    try {
        const response = await fetch(`${API_BASE_URL}/search/compositional`, {
            method: 'POST',
            headers: { 'Content-Type': 'application/json' },
            body: JSON.stringify({ base_image_id: baseImageId, add_text: addText || null, subtract_text: subtractText || null })
        });
        const data = await response.json();
        if(response.ok) { setResults(data.results); setExpandedQuery('Compositional Search applied'); }
        else alert(data.detail);
    } catch (error) { alert("Gagal memproses pencarian komposisional."); }
    setLoading(false);
  };

  return (
    <div className="App" style={{ padding: '40px', fontFamily: 'sans-serif' }}>
      <h1>Advanced E-Commerce Search (NUI)</h1>
      <div style={{ display: 'flex', gap: '40px', marginBottom: '30px' }}>
          <div style={{ flex: 1, padding: '20px', border: '1px solid #ddd', borderRadius: '8px' }}>
            <h3>1. Situational & Semantic Reasoning</h3>
            <input type="text" value={query} onChange={(e) => setQuery(e.target.value)} placeholder="e.g. Black shoes" style={{ width: '100%', padding: '10px', marginBottom: '10px' }} />
            <button onClick={() => executeSemanticSearch(query)} style={{ padding: '10px', width: '100%' }} disabled={loading}>{loading ? 'Processing...' : 'Search'}</button>
          </div>
          <div style={{ flex: 1, padding: '20px', border: '1px solid #ddd', borderRadius: '8px', backgroundColor: '#f9f9f9' }}>
            <h3>2. Compositional Search</h3>
            <input type="text" placeholder="Base Image ID" value={baseImageId} onChange={e=>setBaseImageId(e.target.value)} style={{ width: '100%', marginBottom: '10px', padding: '8px'}}/>
            <input type="text" placeholder="Add (+)" value={addText} onChange={e=>setAddText(e.target.value)} style={{ width: '100%', marginBottom: '10px', padding: '8px'}}/>
            <button onClick={handleCompositionalSearch} style={{ padding: '10px', width: '100%' }} disabled={loading}>Manipulate & Search</button>
          </div>
      </div>
      {expandedQuery && <div style={{ padding: '15px', backgroundColor: '#e6ffe6', marginBottom: '20px' }}><strong>Search Intent: </strong> <i>{expandedQuery}</i></div>}
      <h2>Results Gallery</h2>
      {results.length === 0 && !loading && <p>No results found.</p>}
      <div style={{ display: 'flex', gap: '20px', flexWrap: 'wrap' }}>
        {results.map((res, idx) => (
          <div key={idx} style={{ border: '1px solid #ddd', padding: '10px', borderRadius: '8px', width: '220px' }}>
            <p><strong>ID:</strong> {res.id}</p><p><strong>Score:</strong> {res.score.toFixed(4)}</p>
          </div>
        ))}
      </div>
    </div>
  );
}
export default App;

Overwriting /content/Smart-Catalog-Amazon-Berkeley-Objects/frontend/src/App.tsx


In [8]:
!cd /content/Smart-Catalog-Amazon-Berkeley-Objects/frontend && npm install && npm run build

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋
up to date, audited 1322 packages in 7s
⠙
⠙272 packages are looking for funding
⠙  run `npm fund` for details
⠙
28 vulnerabilities (9 low, 6 moderate, 13 high)

To address issues that do not require attention, run:
  npm audit fix

To address all issues (including breaking changes), run:
  npm audit fix --force

Run `npm audit` for details.
⠹
> frontend@0.1.0 build
> react-scripts build

Creating an optimized production build...
Compiled successfully.

File sizes after gzip:

  62.95 kB  build/static/js/main.e4ddfc6c.js
  1.76 kB   build/static/js/453.20359781.chunk.js
  513 B     build/static/css/main.f855e6bc.css

The project was built assuming it is hosted at /.
You can control this with the homepage field in your package.json.

The build folder is ready to be deployed.
You may serve it with a static server:

  npm install -g serve
  serve -s build

Find out more about deployment here:

  https://cra.link/deployment

⠙

In [6]:
import os
import gzip
import json
import pandas as pd
from PIL import Image
import sys
import warnings
warnings.filterwarnings('ignore')

# 1. Menghubungkan sel ini langsung dengan mesin utama (core.py)
if "/content/Smart-Catalog-Amazon-Berkeley-Objects/backend" not in sys.path:
    sys.path.append("/content/Smart-Catalog-Amazon-Berkeley-Objects/backend")

from core import Embedder, VectorStore, ImageSegmenter

print("=== MEMUAT MESIN AI (Membutuhkan 30-60 detik) ===")
embedder = Embedder()
segmenter = ImageSegmenter()
vector_store = VectorStore()
print("Mesin AI siap dan terhubung ke database fisik!\n")

# --- Step 1 & 2: Load and prepare metadata ---
print("Memuat metadata produk...")
data = []
listing_path = 'dataset/listings/listings/metadata/listings_0.json.gz'

if not os.path.exists(listing_path):
    listing_path = 'dataset/listings/metadata/listings_0.json.gz'

try:
    if listing_path:
        with gzip.open(listing_path, 'rt') as f:
            for i, line in enumerate(f):
                if i >= 30000:
                    break
                try:
                    item = json.loads(line.strip())
                    data.append(item)
                except json.JSONDecodeError:
                    continue
except Exception as e:
    print(f"Error loading metadata: {e}")

df_listings = pd.DataFrame(data)

def extract_en_text(item_list):
    if isinstance(item_list, list):
        for entry in item_list:
            if isinstance(entry, dict) and entry.get('language_tag', '').startswith('en_'):
                return entry.get('value', '')
        if len(item_list) > 0 and isinstance(item_list[0], dict):
            return item_list[0].get('value', '')
    return None

if not df_listings.empty:
    df_listings['item_name_en'] = df_listings['item_name'].apply(extract_en_text)
    if 'product_type' in df_listings.columns:
        df_listings['category'] = df_listings['product_type'].apply(lambda x: x[0]['value'] if isinstance(x, list) and len(x) > 0 else None)
    else:
        df_listings['category'] = None

print("Memuat metadata gambar...")
images_csv_path = 'dataset/images/metadata/images.csv.gz'
try:
    df_images = pd.read_csv(images_csv_path) if os.path.exists(images_csv_path) else pd.DataFrame()
except Exception:
    df_images = pd.DataFrame()

# --- Step 3: Merge, Filter, and Index DIRECTLY ---
df_merged = pd.DataFrame()
if not df_listings.empty and not df_images.empty:
    df_merged = pd.merge(df_listings, df_images, left_on='main_image_id', right_on='image_id', how='inner')
    if 'path' in df_merged.columns:
        df_merged['image_path'] = 'dataset/images/small/' + df_merged['path']
    else:
        df_merged['image_path'] = None

    df_final = df_merged[['item_id', 'image_path', 'item_name_en', 'category']].rename(columns={'item_id': 'id', 'item_name_en': 'text'})
else:
    df_final = pd.DataFrame()

print("\nMulai mengekstrak fitur dan menyimpan ke Vector DB fisik...")
shoe_items_df = pd.DataFrame()
if not df_final.empty and 'category' in df_final.columns:
    shoe_items_df = df_final[
        (df_final['category'].str.contains('shoes', case=False, na=False)) |
        (df_final['category'].str.contains('footwear', case=False, na=False))
    ].copy()

total_items = len(shoe_items_df) if not shoe_items_df.empty else 0
indexed_count = 0
failed_count = 0

if total_items > 0:
    for i, row in shoe_items_df.iterrows():
        img_id = str(row['id'])
        image_local_path = row['image_path']

        if (i + 1) % 50 == 0 or i == 0 or (i + 1) == total_items:
            print(f"Memproses {i+1}/{total_items}: ID={img_id}")

        try:
            if not os.path.exists(image_local_path):
                failed_count += 1
                continue

            # Bypass server: Proses gambar dan masukkan langsung ke database dengan Python
            image = Image.open(image_local_path).convert("RGB")
            segmented_image = segmenter.segment_shoe(image)
            vector = embedder.get_image_embedding(segmented_image)

            vector_store.insert_image(
                id=img_id,
                vector=vector,
                payload={"original_filename": f"{img_id}.jpg"}
            )

            indexed_count += 1

        except Exception as e:
            print(f"Gagal memproses {img_id}: {e}")
            failed_count += 1
else:
    print("Tidak ada item sepatu untuk diindeks.")

print(f"\n--- Indeksasi Fisik Selesai ---")
print(f"Berhasil diindeks: {indexed_count}")
print(f"Gagal diindeks: {failed_count}")

Using device: cpu
=== MEMUAT MESIN AI (Membutuhkan 30-60 detik) ===


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Mesin AI siap dan terhubung ke database fisik!

Memuat metadata produk...
Memuat metadata gambar...

Mulai mengekstrak fitur dan menyimpan ke Vector DB fisik...
Memproses 1/761: ID=B06X9STHNG
Memproses 50/761: ID=B084LLQH9N
Memproses 150/761: ID=B075T4DC11
Memproses 200/761: ID=B07N295F7B
Memproses 1950/761: ID=B07XFW7JY8
Memproses 2100/761: ID=B07XLB83F4
Memproses 2850/761: ID=B07357QJHQ
Memproses 4350/761: ID=B07R58BL8Q
Memproses 4500/761: ID=B07VSFL77Q
Memproses 4550/761: ID=B082PMX2R2
Memproses 5900/761: ID=B07RW5HYN6
Memproses 6500/761: ID=B07RBLM8J1
Memproses 6850/761: ID=B07TJ9JXH3
Memproses 7500/761: ID=B074DXZ9XB
Memproses 8050/761: ID=B072V7HFX4
Memproses 8150/761: ID=B085D5K3GP
Memproses 8300/761: ID=B082QLFPVF
Memproses 8350/761: ID=B07RB4PRH6

--- Indeksasi Fisik Selesai ---
Berhasil diindeks: 761
Gagal diindeks: 0


In [10]:
import os
import time
import re
import subprocess

print("1. Menginstal ulang Cloudflare Tunnel (karena sesi sempat di-restart)...")
if not os.path.exists("/usr/bin/cloudflared"):
    os.system("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb")
    os.system("dpkg -i cloudflared-linux-amd64.deb")

print("2. Menyalakan Uvicorn Server Utama...")
os.system("fuser -k 8000/tcp || true; pkill -f cloudflared || true; pkill -f uvicorn || true")
os.system("cd /content/Smart-Catalog-Amazon-Berkeley-Objects/backend && nohup uvicorn api:app --host 0.0.0.0 --port 8000 > uvicorn.log 2>&1 &")
time.sleep(8)

print("3. Membuka jalur Cloudflare...")
cf_log = "cloudflare.log"
if os.path.exists(cf_log):
    os.remove(cf_log)

os.system(f"nohup cloudflared tunnel --url http://localhost:8000 > {cf_log} 2>&1 &")

url = ""
for _ in range(30):
    time.sleep(1)
    if os.path.exists(cf_log):
        with open(cf_log, "r") as f:
            log_content = f.read()
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', log_content)
            if match:
                url = match.group(0)
                break

if url:
    print("\n" + "="*70)
    print("✅ SERVER SIAP! DATA 761 SEPATU SUDAH MASUK.")
    print(f"👉 BUKA WEB ANDA DI SINI: {url}")
    print("="*70 + "\n")
else:
    print("\n❌ Gagal mendapatkan URL. Ini isi log error-nya untuk pengecekan:")
    if os.path.exists(cf_log):
        with open(cf_log, "r") as f:
            print(f.read())

1. Menginstal ulang Cloudflare Tunnel (karena sesi sempat di-restart)...
2. Menyalakan Uvicorn Server Utama...
3. Membuka jalur Cloudflare...

✅ SERVER SIAP! DATA 761 SEPATU SUDAH MASUK.
👉 BUKA WEB ANDA DI SINI: https://jan-finish-cleveland-fixed.trycloudflare.com



In [11]:
import os
import re
import time
import subprocess

print("1. Mematikan server yang menahan memori kosong...")
os.system("fuser -k 8000/tcp || true; pkill -f cloudflared || true; pkill -f uvicorn || true")
time.sleep(2)

print("2. Memastikan saringan threshold benar-benar dimatikan di api.py...")
api_path = "/content/Smart-Catalog-Amazon-Berkeley-Objects/backend/api.py"
with open(api_path, "r") as f:
    content = f.read()

# Memaksa pencarian mengembalikan 5 hasil berapapun skor kemiripannya
content = re.sub(r'threshold=TRIAL_228_CONFIG\["similarity_threshold"\]', 'threshold=None', content)
with open(api_path, "w") as f:
    f.write(content)

print("3. Menyalakan ulang Uvicorn (Membaca 761 data dari folder!)...")
os.system("cd /content/Smart-Catalog-Amazon-Berkeley-Objects/backend && nohup uvicorn api:app --host 0.0.0.0 --port 8000 > uvicorn.log 2>&1 &")
time.sleep(5)

print("4. Membuka jalur Cloudflare...")
if os.path.exists("cloudflare.log"):
    os.remove("cloudflare.log")

os.system("nohup cloudflared tunnel --url http://localhost:8000 > cloudflare.log 2>&1 &")

url = ""
for _ in range(30):
    time.sleep(1)
    if os.path.exists("cloudflare.log"):
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', open("cloudflare.log", "r").read())
        if match:
            url = match.group(0)
            break

if url:
    print("\n" + "="*70)
    print("✅ RESTART BERHASIL! SERVER KINI MEMBACA 761 DATA ANDA.")
    print(f"👉 BUKA WEB ANDA DI SINI: {url}")
    print("="*70)
    print("Silakan klik tautan di atas dan cari 'Black shoes'. Hasilnya PASTI muncul.")
else:
    print("\n❌ Gagal mendapatkan URL. Periksa log sistem.")

1. Mematikan server yang menahan memori kosong...
2. Memastikan saringan threshold benar-benar dimatikan di api.py...
3. Menyalakan ulang Uvicorn (Membaca 761 data dari folder!)...
4. Membuka jalur Cloudflare...

✅ RESTART BERHASIL! SERVER KINI MEMBACA 761 DATA ANDA.
👉 BUKA WEB ANDA DI SINI: https://ballet-pot-largest-arizona.trycloudflare.com
Silakan klik tautan di atas dan cari 'Black shoes'. Hasilnya PASTI muncul.
